## 📌 Executive Summary

Customer churn directly impacts revenue and growth. This analysis explores behavioral and contractual patterns that drive churn and prepares a high-quality dataset for downstream machine learning.

Key outcomes:
- Identification of high-risk customer segments
- Creation of predictive features
- Preparation of a clean, model-ready dataset

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 📥 1. Import & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

# 📂 2. Data Ingestion
👉 Use Kaggle Telco Churn dataset

In [ ]:
DATA_PATH = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
df.head()

# 🔍 3. Data Audit

In [ ]:
print("🔍 Data Types:")
display(df.dtypes)

In [ ]:
print("\n📊 Missing Values:")
display(df.isnull().sum())

In [ ]:
print("\n📊 Duplicate Rows:", df.duplicated().sum())

# 🧹 4. Data Cleaning

In [ ]:
# Convert TotalCharges safely
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop missing
df = df.dropna().copy()

# Drop ID column
df.drop(columns=["customerID"], inplace=True)

print("Post-cleaning shape:", df.shape)
df.head()

# 📊 5. Target Variable Analysis

In [ ]:
churn_counts = df["Churn"].value_counts()

print(churn_counts)

sns.countplot(x="Churn", data=df)
plt.title("Customer Churn Distribution")
plt.show()

### Insight
- Evaluate class imbalance before modeling  
- Important for selecting evaluation metrics later  

# 📊 6. Univariate Analysis

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

df[num_cols].describe()

In [ ]:
df[num_cols].hist(bins=30, figsize=(10,6))
plt.tight_layout()
plt.show()

# 📊 7. Categorical Feature Analysis

In [ ]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    print(f"\n🔹 {col}")
    display(df[col].value_counts())

# 📊 8. Churn Drivers (Bivariate Analysis)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="Contract", hue="Churn", data=df)
plt.title("Churn by Contract Type")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

### Key Observations
- Month-to-month contracts show higher churn  
- Higher monthly charges correlate with churn  

# ⚙️ 9. Feature Engineering

In [ ]:
# Tenure segmentation
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

# Spending behavior
df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)

# Customer loyalty flag
df["IsLongTerm"] = (df["tenure"] > 24).astype(int)

# High value customer
df["HighValueCustomer"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)

df.head()

# 🔢 10. Encoding (Minimal for next step)

In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df["Churn"].value_counts())

# 📊 11. Correlation Analysis

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

# 💾 12. Save Processed Data 

In [ ]:
OUTPUT_PATH = Path("processed_churn.csv")

df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Processed dataset saved at: {OUTPUT_PATH}")

## 🧠 Key Insights

- Customers with shorter tenure are more likely to churn  
- Month-to-month contracts show higher churn rates  
- Higher monthly charges correlate with increased churn  
- Long-term customers are more stable  

---

## 🏁 Conclusion

In this notebook, we explored customer churn data and created meaningful features to better understand customer behavior.

The processed dataset is now ready for machine learning modeling in the next step.

---